# LSTM V3 — Kaggle ready (.zip.bin supported)

Import notebook này bằng **File → Import Notebook**. Attach dataset V3 đã cập nhật, chọn GPU rồi chạy lần lượt hoặc Run All. Không paste JSON notebook vào code cell.

Dataset cần CSV, dataset.manifest.json, protocol.json, train_lstm_kaggle_v3.py và runtime ZIP gốc hoặc ZIP có đuôi .zip.bin. Thư mục Kaggle tự giải nén không thay thế file ZIP có checksum. Notebook giữ nguyên kiểm tra checksum, không sửa dataset input hoặc V2.

Mặc định chạy pilot 3 epochs; muốn train đầy đủ, đổi RUN_MODE thành full ở cell cấu hình. Chỉ upload dữ liệu nếu quyền nguồn cho phép. Model train xong không tự đủ điều kiện promote.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

RUN_MODE = 'pilot'  # 'pilot' = smoke 3 epochs; 'full' = 3 folds x 2 seeds, max 25 epochs/trial
BATCH_SIZE = 256
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working')
# Nếu attach nhiều dataset V3, điền đường dẫn CSV cụ thể để tránh chọn nhầm.
DATASET_CSV = ''
# Để trống cho run mới. Khi resume, điền đúng thư mục run cũ và giữ nguyên mode/batch.
RESUME_RUN_DIR = ''

if RUN_MODE not in {'pilot', 'full'} or not isinstance(BATCH_SIZE, int) or BATCH_SIZE < 1:
    raise ValueError('RUN_MODE phải là pilot/full và BATCH_SIZE là số nguyên dương.')
if not INPUT_ROOT.is_dir() or not OUTPUT_ROOT.is_dir():
    raise RuntimeError('Notebook này dành cho Kaggle; cần /kaggle/input và /kaggle/working.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-' + uuid4().hex[:8]
print('Mode:', RUN_MODE, '| run:', RUN_ID)


In [ ]:
import hashlib
import json
import re
import shutil
import zipfile

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def prepare_v3_input(input_root, staging_root, dataset_csv=''):
    input_root = Path(input_root).resolve()
    if dataset_csv:
        csv = Path(dataset_csv).resolve()
        if not csv.is_relative_to(input_root) or not csv.is_file():
            raise ValueError('DATASET_CSV phải là file thuộc dataset input đã attach.')
    else:
        matches = sorted(input_root.rglob('vci_ohlcv_lstm_v3.csv'))
        if len(matches) != 1:
            raise FileNotFoundError(f'Tìm thấy {len(matches)} CSV V3. Attach đúng dataset hoặc điền DATASET_CSV.')
        csv = matches[0].resolve()
    source = csv.parent
    required = ['dataset.manifest.json', 'protocol.json', 'train_lstm_kaggle_v3.py']
    for name in required:
        if not (source / name).is_file():
            raise FileNotFoundError(f'Thiếu {name} cùng thư mục CSV: {source}')
    manifest = json.loads((source / 'dataset.manifest.json').read_text(encoding='utf-8'))
    bundle_info = manifest['runtime_bundle']
    filename = bundle_info['filename']
    expected = bundle_info['sha256']
    if not isinstance(filename, str) or '/' in filename or chr(92) in filename or Path(filename).name != filename or not filename.endswith('.zip'):
        raise ValueError('Tên runtime ZIP trong manifest không an toàn.')
    if not isinstance(expected, str) or not re.fullmatch('[0-9a-fA-F]{64}', expected):
        raise ValueError('Manifest thiếu SHA-256 hợp lệ của runtime ZIP.')
    candidates = [source / filename, source / (filename + '.bin')]
    candidates = [path for path in candidates if path.is_file()]
    if not candidates:
        raise FileNotFoundError(f'Thiếu {filename} hoặc {filename}.bin. Kaggle tự giải nén ZIP: upload thêm bản .zip.bin, lưu version và cập nhật notebook input.')
    verified = [path for path in candidates if file_sha256(path) == expected.lower()]
    if not verified:
        raise ValueError('Runtime ZIP checksum không khớp manifest. Upload đúng ZIP gốc; không sửa checksum để bỏ kiểm tra.')
    archive = verified[0]
    if not zipfile.is_zipfile(archive):
        raise ValueError('Runtime không phải ZIP hợp lệ.')
    with zipfile.ZipFile(archive) as runtime:
        if runtime.testzip() is not None:
            raise ValueError('Runtime ZIP bị hỏng CRC.')
    csv_info = manifest['files']['dataset']
    if csv_info['filename'] != csv.name or file_sha256(csv) != csv_info['sha256']:
        raise ValueError('CSV không khớp dataset.manifest.json; cần dùng cùng phiên bản dữ liệu/manifest.')
    protocol = json.loads((source / 'protocol.json').read_text(encoding='utf-8'))
    if protocol.get('window') != 30 or protocol.get('horizons') != [1, 5]:
        raise ValueError('Sai protocol V3 window/horizons.')
    stage = Path(staging_root).resolve()
    if stage == input_root or stage.is_relative_to(input_root):
        raise ValueError('Staging phải nằm ngoài input chỉ đọc.')
    stage.mkdir(parents=True, exist_ok=False)
    csv_target = stage / csv.name
    csv_target.symlink_to(csv)  # Kaggle Linux: không copy lại CSV 525 MB.
    for name in required:
        shutil.copy2(source / name, stage / name)
    shutil.copy2(archive, stage / filename)
    if file_sha256(stage / filename) != expected.lower():
        raise ValueError('ZIP sau staging không khớp checksum.')
    print('CSV đã verify:', csv.name, '| bytes:', csv.stat().st_size)
    print('Runtime đã verify:', filename, '| input:', archive.name)
    print('Rows:', manifest.get('retained_rows'), '| tickers:', manifest.get('tickers'), '| endpoints:', manifest.get('training_endpoints'))
    return csv_target, stage / 'train_lstm_kaggle_v3.py'


In [ ]:
import runpy
STAGING_ROOT = OUTPUT_ROOT / ('v3-input-' + RUN_ID)
CSV_PATH, SCRIPT_PATH = prepare_v3_input(INPUT_ROOT, STAGING_ROOT, DATASET_CSV)
training = runpy.run_path(str(SCRIPT_PATH), run_name='v3_training_import')
# Verify/import shared runtime trước khi tải TensorFlow; không dùng thư mục ZIP tự giải nén.
training['_bootstrap_runtime'](CSV_PATH)
DATA_ARGS = ['--data', str(CSV_PATH)]
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__, '| GPUs:', gpus)
if not gpus:
    raise RuntimeError('Chọn Settings → Accelerator → GPU rồi chạy lại session.')
print('Dữ liệu/runtime sẵn sàng; chưa bắt đầu training.')


In [ ]:
if RESUME_RUN_DIR:
    RUN_DIR = Path(RESUME_RUN_DIR).resolve()
    if not RUN_DIR.is_relative_to(OUTPUT_ROOT.resolve()) or not (RUN_DIR / 'run-config.json').is_file():
        raise ValueError('Resume cần run folder cũ có run-config.json dưới /kaggle/working.')
else:
    RUN_DIR = OUTPUT_ROOT / ('lstm-v3-' + RUN_MODE + '-' + RUN_ID)
arguments = DATA_ARGS + ['--batch-size', str(BATCH_SIZE), '--output', str(RUN_DIR)]
if RUN_MODE == 'pilot':
    arguments += ['--epochs', '3', '--seeds', '42', '--pilot-endpoints', '50000']
if RESUME_RUN_DIR:
    arguments += ['--resume']
print('Output:', RUN_DIR)
print('Arguments:', arguments)
training['main'](arguments)


In [ ]:
from IPython.display import FileLink, display
report_path = RUN_DIR / 'evaluation.json'
if not report_path.is_file():
    raise FileNotFoundError('Chưa có evaluation.json: xem lỗi training ở cell trước, không giả kết quả hoàn tất.')
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report.get('gate', {}), ensure_ascii=False, indent=2))
print('Pilot:', report.get('pilot'), '| scope:', report.get('scope'))
RESULT_ZIP = OUTPUT_ROOT / (RUN_DIR.name + '-results.zip')
if RESULT_ZIP.exists():
    raise FileExistsError('Archive đã tồn tại; dùng kết quả cũ hoặc chọn run mới, không overwrite.')
shutil.make_archive(str(RESULT_ZIP.with_suffix('')), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('Tải archive ở Output hoặc link bên dưới. Trên máy lưu vào ổ D, không ghi đè V2.')
display(FileLink(str(RESULT_ZIP.relative_to(OUTPUT_ROOT))))
print('Archive:', RESULT_ZIP)
print('Training hoàn tất không đồng nghĩa model đã qua gate hoặc được phép đổi Technical score.')
